# CIFAR-10 ResNet 训练 Notebook

这个 notebook 从零实现一个适合 CIFAR-10 的 ResNet，并把前向传播、训练流程和验证流程拆成独立的代码单元展示。训练器部分补齐了优化器、LR 调度、AMP 混合精度、梯度裁剪、checkpoint 保存/恢复，以及随机种子和可复现设置。整个流程使用 `click` 解析命令行参数，使用 `logging` 记录训练日志，便于在 Notebook 和终端里统一调试。

## 1. 目标与流程

1. 先把原始 CIFAR-10 batch 导出成按类别整理好的图片文件夹。
2. 再扫描图片目录生成 `train.txt` 和 `val.txt` 索引文件，并用索引驱动数据读取。
3. 定义适合 CIFAR-10 的 ResNet。
4. 将 forward、train、val 分别放在独立代码单元中展示。
5. 训练器包含优化器、LR 调度、AMP、梯度裁剪和 checkpoint 恢复。
6. 用 `click` 提供命令行参数，用 `logging` 输出训练过程。

In [17]:
from __future__ import annotations

import csv
import logging
import pickle
import random
from pathlib import Path
from typing import Iterable

import click
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

DEFAULT_WORKSPACE_DIR = Path.cwd().resolve()
DEFAULT_CIFAR10_DIR = DEFAULT_WORKSPACE_DIR / Path("CiFar10")
DEFAULT_INPUT_DIR = DEFAULT_CIFAR10_DIR / Path("cifar-10-batches-py")
DEFAULT_EXPORT_DIR = DEFAULT_INPUT_DIR / Path("export")
DEFAULT_INDEX_DIR = DEFAULT_CIFAR10_DIR / Path("dataset_index")
DEFAULT_LOG_DIR = DEFAULT_CIFAR10_DIR / Path("Log")
DEFAULT_LOG_NAME = "resnet_cifar10.log"
DEFAULT_CHECKPOINT_PATH = DEFAULT_LOG_DIR / Path("resnet_cifar10_checkpoint.pth")
DEFAULT_HISTORY_CSV_PATH = DEFAULT_LOG_DIR / Path("training_history.csv")
DEFAULT_HISTORY_PLOT_PATH = DEFAULT_LOG_DIR / Path("training_history.png")
DEFAULT_SEED = 42
DEFAULT_EPOCHS = 10
DEFAULT_BATCH_SIZE = 128
DEFAULT_IMAGE_SIZE = 32
DEFAULT_NUM_WORKERS = 0
DEFAULT_LR = 0.1
DEFAULT_WEIGHT_DECAY = 5e-4
DEFAULT_TRAIN_RATIO = 0.9
DEFAULT_PATTERN = "data_batch_*"
DEFAULT_INCLUDE_TEST = False
DEFAULT_USE_AMP = True
DEFAULT_GRAD_CLIP_NORM = 1.0
DEFAULT_MEAN = (0.4914, 0.4822, 0.4465)
DEFAULT_STD = (0.2470, 0.2435, 0.2616)


def set_seed(seed: int = DEFAULT_SEED) -> None:
    # 固定随机种子，保证数据切分、参数初始化和训练过程尽量可复现。
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_device() -> torch.device:
    # 优先使用 GPU；没有 GPU 时自动退回 CPU。
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def setup_logging(log_dir: Path = DEFAULT_LOG_DIR, log_name: str = DEFAULT_LOG_NAME) -> logging.Logger:
    log_dir.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger("resnet_cifar10")
    logger.setLevel(logging.INFO)
    logger.propagate = False

    for handler in list(logger.handlers):
        logger.removeHandler(handler)
        handler.close()

    formatter = logging.Formatter("%(asctime)s %(levelname)s %(message)s")
    file_handler = logging.FileHandler(log_dir / log_name, encoding="utf-8")
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)

    stream_handler = logging.StreamHandler()
    stream_handler.setFormatter(formatter)
    logger.addHandler(stream_handler)

    return logger

## 2. CIFAR-10 批次导出

这一格负责把原始 `cifar-10-batches-py` 里的 batch 文件展开为按类别分组的图片目录，便于后续按目录生成索引并读取。

In [18]:
def load_meta(meta_path: Path) -> list[str]:
    with meta_path.open("rb") as handle:
        data = pickle.load(handle, encoding="latin1")

    label_names = data.get(b"label_names") or data.get("label_names")
    if label_names is None:
        raise ValueError(f"Missing label_names in {meta_path}")

    return [name.decode("utf-8") if isinstance(name, bytes) else str(name) for name in label_names]

In [19]:
def load_batch(batch_path: Path) -> dict:
    with batch_path.open("rb") as handle:
        return pickle.load(handle, encoding="latin1")


def unpack_batch(batch: dict) -> tuple[object, object, object]:
    images = batch.get(b"data") or batch.get("data")
    labels = batch.get(b"labels") or batch.get("labels")
    filenames = batch.get(b"filenames") or batch.get("filenames")

    if images is None or labels is None or filenames is None:
        raise ValueError("Unexpected CIFAR batch format")

    return images, labels, filenames

In [20]:
def export_batches_by_class(
    input_dir: Path,
    output_dir: Path,
    pattern: str = DEFAULT_PATTERN,
    include_test: bool = False,
    logger: logging.Logger | None = None,
) -> dict[str, int]:
    logger = logger or logging.getLogger("resnet_cifar10")
    class_names = load_meta(input_dir / "batches.meta")
    batch_paths = sorted(input_dir.glob(pattern))

    if include_test:
        test_batch = input_dir / "test_batch"
        if test_batch.exists() and test_batch not in batch_paths:
            batch_paths.append(test_batch)

    logger.info(
        "开始导出 CIFAR-10 图片: input_dir=%s output_dir=%s pattern=%s include_test=%s",
        input_dir,
        output_dir,
        pattern,
        include_test,
    )

    counts: dict[str, int] = {}

    for batch_path in batch_paths:
        logger.info("处理批次文件: %s", batch_path.name)
        batch = load_batch(batch_path)
        images, labels, filenames = unpack_batch(batch)

        for flat_image, label, filename in zip(images, labels, filenames):
            class_name = class_names[int(label)]
            class_dir = output_dir / class_name
            class_dir.mkdir(parents=True, exist_ok=True)

            image = flat_image.reshape(3, 32, 32).transpose(1, 2, 0)
            image_name = filename.decode("utf-8") if isinstance(filename, bytes) else str(filename)
            output_path = class_dir / f"{Path(image_name).stem}.png"
            Image.fromarray(image).save(output_path)
            counts[class_name] = counts.get(class_name, 0) + 1

    total = sum(counts.values())
    logger.info("导出完成，共写入 %s 张图片到 %s", total, output_dir)
    for class_name in sorted(counts):
        logger.info("%s: %s", class_name, counts[class_name])

    return counts

In [21]:
def build_and_save_indices(
    root_dir: Path = DEFAULT_EXPORT_DIR,
    output_dir: Path = DEFAULT_INDEX_DIR,
    seed: int = DEFAULT_SEED,
    logger: logging.Logger | None = None,
) -> dict[str, int]:
    root_dir = root_dir.resolve()
    output_dir = output_dir.resolve()
    logger = logger or logging.getLogger("resnet_cifar10")
    logger.info("开始生成索引文件: root_dir=%s output_dir=%s seed=%s", root_dir, output_dir, seed)
    samples = collect_samples(root_dir)
    logger.info("扫描到图片总数: %s", len(samples))
    train_samples, val_samples = split_class_samples(samples, seed=seed)
    logger.info("划分结果: train=%s val=%s", len(train_samples), len(val_samples))

    test_index = output_dir / "test.txt"
    if test_index.exists():
        test_index.unlink()
        logger.info("已删除旧的 test 索引文件: %s", test_index)

    counts = {
        "train": write_index_file(output_dir / "train.txt", train_samples),
        "val": write_index_file(output_dir / "val.txt", val_samples),
    }
    logger.info("索引文件写入完成: %s", counts)
    return counts


def is_image_file(path: Path) -> bool:
    return path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def collect_class_folders(root_dir: Path) -> list[Path]:
    return sorted([path for path in root_dir.iterdir() if path.is_dir()])


def collect_samples(root_dir: Path) -> list[tuple[Path, int, str]]:
    class_folders = collect_class_folders(root_dir)
    samples: list[tuple[Path, int, str]] = []

    for class_index, class_folder in enumerate(class_folders):
        for image_path in sorted(class_folder.rglob("*")):
            if is_image_file(image_path):
                relative_path = image_path.relative_to(root_dir).as_posix()
                samples.append((image_path, class_index, relative_path))

    return samples


def split_class_samples(samples: list[tuple[Path, int, str]], seed: int = DEFAULT_SEED) -> tuple[list[tuple[Path, int, str]], list[tuple[Path, int, str]]]:
    rng = random.Random(seed)
    grouped_samples: dict[int, list[tuple[Path, int, str]]] = {}
    for sample in samples:
        grouped_samples.setdefault(sample[1], []).append(sample)

    train_samples: list[tuple[Path, int, str]] = []
    val_samples: list[tuple[Path, int, str]] = []

    for class_index in sorted(grouped_samples):
        class_samples = grouped_samples[class_index][:]
        rng.shuffle(class_samples)
        total = len(class_samples)
        train_count = int(total * DEFAULT_TRAIN_RATIO)
        val_count = total - train_count

        train_samples.extend(class_samples[:train_count])
        val_samples.extend(class_samples[train_count:train_count + val_count])

    return train_samples, val_samples


def write_index_file(index_path: Path, samples: Iterable[tuple[Path, int, str]]) -> int:
    index_path.parent.mkdir(parents=True, exist_ok=True)
    count = 0
    with index_path.open("w", encoding="utf-8") as handle:
        for _, class_index, relative_path in samples:
            handle.write(f"{relative_path}\t{class_index}\n")
            count += 1
    return count

In [22]:
def prepare_data_pipeline(
    input_dir: Path = DEFAULT_INPUT_DIR,
    export_dir: Path = DEFAULT_EXPORT_DIR,
    index_dir: Path = DEFAULT_INDEX_DIR,
    pattern: str = DEFAULT_PATTERN,
    include_test: bool = DEFAULT_INCLUDE_TEST,
    seed: int = DEFAULT_SEED,
    logger: logging.Logger | None = None,
) -> tuple[dict[str, int], dict[str, int]]:
    logger = logger or logging.getLogger("resnet_cifar10")
    export_counts = export_batches_by_class(
        input_dir=input_dir,
        output_dir=export_dir,
        pattern=pattern,
        include_test=include_test,
        logger=logger,
    )
    index_counts = build_and_save_indices(
        root_dir=export_dir,
        output_dir=index_dir,
        seed=seed,
        logger=logger,
    )
    return export_counts, index_counts

## 3. ResNet 基础模块

这一部分定义残差块和 ResNet 主体。ResNet 的整体前向路径会放在下一格单独展示，这一格只负责模型结构本身。

In [23]:
def conv3x3(in_channels: int, out_channels: int, stride: int = 1) -> nn.Conv2d:
    # 3x3 卷积是 ResNet 的核心卷积核，padding=1 保持空间尺寸对齐。
    return nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels: int, out_channels: int, stride: int = 1) -> None:
        super().__init__()
        self.conv1 = conv3x3(in_channels, out_channels, stride=stride)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = conv3x3(out_channels, out_channels)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # 当通道数或步幅变化时，残差分支也需要同步变换尺寸。
        self.downsample = None
        if stride != 1 or in_channels != out_channels * self.expansion:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels * self.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels * self.expansion),
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = F.relu(out, inplace=True)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out = out + identity
        out = F.relu(out, inplace=True)
        return out


class ResNet(nn.Module):
    def __init__(self, block: type[BasicBlock], layers: tuple[int, int, int, int], num_classes: int = 10) -> None:
        super().__init__()

        # CIFAR-10 图像较小，这里使用更适合 32x32 输入的 stem。
        self.in_channels = 64
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )

        self.layer1 = self._make_layer(block, 64, layers[0], stride=1)
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block: type[BasicBlock], out_channels: int, blocks: int, stride: int) -> nn.Sequential:
        layers = [block(self.in_channels, out_channels, stride=stride)]
        self.in_channels = out_channels * block.expansion

        for _ in range(1, blocks):
            layers.append(block(self.in_channels, out_channels, stride=1))

        return nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 真正的 forward 路径在下一格单独展开，这里只做函数转发。
        return resnet_forward(self, x)


def resnet18(num_classes: int = 10) -> ResNet:
    # 一个轻量的 ResNet-18 变体，适合 CIFAR-10 这种小图分类任务。
    return ResNet(BasicBlock, (2, 2, 2, 2), num_classes=num_classes)

## 4. Forward 流程

这一格只展示 ResNet 的前向传播路径：先经过 stem，再依次经过四个残差阶段，最后做全局平均池化和全连接分类。

In [24]:
def resnet_forward(model: ResNet, x: torch.Tensor) -> torch.Tensor:
    # 1. Stem：把原始 RGB 图像映射到较高维的特征空间。
    x = model.stem(x)

    # 2. 四个残差阶段：每个阶段逐步提取更抽象的语义特征。
    x = model.layer1(x)
    x = model.layer2(x)
    x = model.layer3(x)
    x = model.layer4(x)

    # 3. 全局平均池化：把空间维度压缩到 1x1，减少参数量。
    x = model.avgpool(x)

    # 4. 展平后送入分类头，输出每个类别的 logits。
    x = torch.flatten(x, 1)
    logits = model.fc(x)
    return logits

## 5. 索引与数据加载

这里读取前面生成的 `train.txt` 和 `val.txt`，用 `ImageIndexDataset` 按需加载图片，并把训练集和验证集分别接上不同的数据增强。

In [25]:
def build_transforms(image_size: int = DEFAULT_IMAGE_SIZE) -> tuple[transforms.Compose, transforms.Compose]:
    # 训练集加入随机增强，帮助模型提升泛化能力。
    train_transform = transforms.Compose([
        transforms.RandomCrop(image_size, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
        transforms.ToTensor(),
        transforms.Normalize(DEFAULT_MEAN, DEFAULT_STD),
    ])

    # 验证集不做随机操作，只保留标准化。
    eval_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(DEFAULT_MEAN, DEFAULT_STD),
    ])

    return train_transform, eval_transform


class ImageIndexDataset(Dataset):
    # 这个包装器只负责把索引文件中的相对路径映射回图片文件。
    def __init__(self, index_file: Path, root_dir: Path | None = None, transform=None, target_transform=None):
        self.index_file = Path(index_file)
        self.root_dir = Path(root_dir) if root_dir is not None else self.index_file.parent
        self.transform = transform
        self.target_transform = target_transform
        self.samples: list[tuple[Path, int]] = []

        with self.index_file.open("r", encoding="utf-8") as handle:
            for line in handle:
                line = line.strip()
                if not line:
                    continue
                relative_path, label_text = line.split("\t")
                image_path = Path(relative_path)
                if not image_path.is_absolute():
                    image_path = self.root_dir / image_path
                self.samples.append((image_path, int(label_text)))

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int):
        image_path, label = self.samples[index]
        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)
        if self.target_transform is not None:
            label = self.target_transform(label)

        return image, label


def build_dataloaders(
    image_root_dir: Path = DEFAULT_EXPORT_DIR,
    index_dir: Path = DEFAULT_INDEX_DIR,
    batch_size: int = DEFAULT_BATCH_SIZE,
    image_size: int = DEFAULT_IMAGE_SIZE,
    num_workers: int = DEFAULT_NUM_WORKERS,
    logger: logging.Logger | None = None,
) -> tuple[DataLoader, DataLoader, list[str]]:
    index_dir = index_dir.resolve()
    image_root_dir = image_root_dir.resolve()
    train_index = index_dir / "train.txt"
    val_index = index_dir / "val.txt"

    if not train_index.exists() or not val_index.exists():
        raise FileNotFoundError(f"Index files not found in {index_dir}; please generate them first.")

    train_transform, eval_transform = build_transforms(image_size=image_size)
    train_dataset = ImageIndexDataset(train_index, root_dir=image_root_dir, transform=train_transform)
    val_dataset = ImageIndexDataset(val_index, root_dir=image_root_dir, transform=eval_transform)
    class_names = sorted({sample[0].parent.name for sample in train_dataset.samples + val_dataset.samples})

    pin_memory = torch.cuda.is_available()
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin_memory)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)

    logger = logger or logging.getLogger("resnet_cifar10")
    logger.info("已加载索引数据集: image_root_dir=%s index_dir=%s", image_root_dir, index_dir)
    logger.info("数据集划分完成: train=%s val=%s", len(train_dataset), len(val_dataset))
    logger.info("类别数量: %s", len(class_names))

    return train_loader, val_loader, class_names


def accuracy_from_logits(logits: torch.Tensor, targets: torch.Tensor) -> float:
    predictions = logits.argmax(dim=1)
    return (predictions == targets).float().mean().item()

## 5. Train 流程

这一格放训练一个 epoch 的逻辑，包含前向、反向、AMP 混合精度、梯度裁剪和参数更新。checkpoint 的保存与恢复也在训练器这一部分统一管理。

In [26]:
def save_checkpoint(
    checkpoint_path: Path,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler.LRScheduler | None,
    scaler: GradScaler | None,
    epoch: int,
    best_val_acc: float,
    logger: logging.Logger | None = None,
 ) -> None:
    # 统一保存可恢复训练所需的全部状态。
    logger = logger or logging.getLogger("resnet_cifar10")
    checkpoint_path = Path(checkpoint_path)
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
            "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
            "best_val_acc": best_val_acc,
        },
        checkpoint_path,
    )
    logger.info("已保存 checkpoint: %s (best_val_acc=%.4f)", checkpoint_path, best_val_acc)


def load_checkpoint(
    checkpoint_path: Path,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler.LRScheduler | None,
    scaler: GradScaler | None,
    device: torch.device,
    logger: logging.Logger | None = None,
 ) -> tuple[int, float]:
    # 从 checkpoint 恢复模型、优化器、调度器和 AMP 状态。
    logger = logger or logging.getLogger("resnet_cifar10")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    if scheduler is not None and checkpoint.get("scheduler_state_dict") is not None:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    if scaler is not None and checkpoint.get("scaler_state_dict") is not None:
        scaler.load_state_dict(checkpoint["scaler_state_dict"])

    start_epoch = int(checkpoint.get("epoch", 0)) + 1
    best_val_acc = float(checkpoint.get("best_val_acc", 0.0))
    logger.info("已恢复 checkpoint: %s (start_epoch=%s best_val_acc=%.4f)", checkpoint_path, start_epoch, best_val_acc)
    return start_epoch, best_val_acc


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    scaler: GradScaler | None = None,
    use_amp: bool = True,
    max_grad_norm: float = 1.0,
    logger: logging.Logger | None = None,
 ) -> tuple[float, float]:
    model.train()
    logger = logger or logging.getLogger("resnet_cifar10")
    amp_enabled = use_amp and device.type == "cuda"

    running_loss = 0.0
    running_correct = 0.0
    sample_count = 0

    for step, (images, targets) in enumerate(loader, start=1):
        images = images.to(device)
        targets = targets.to(device)

        # set_to_none=True 可以减少一些无效写入，训练时更高效。
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=amp_enabled):
            logits = model(images)
            loss = criterion(logits, targets)

        if amp_enabled and scaler is not None:
            scaler.scale(loss).backward()
            if max_grad_norm > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            if max_grad_norm > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()

        batch_size = targets.size(0)
        running_loss += loss.item() * batch_size
        running_correct += (logits.argmax(dim=1) == targets).sum().item()
        sample_count += batch_size

        if step % 100 == 0:
            logger.info("train step=%s loss=%.4f", step, loss.item())

    average_loss = running_loss / sample_count
    average_acc = running_correct / sample_count
    logger.info("train epoch done: loss=%.4f acc=%.4f", average_loss, average_acc)
    return average_loss, average_acc

## 6. Val 流程

这一格放验证逻辑：关闭梯度、切换到评估模式，并在同一 AMP 开关下统计验证集上的损失和准确率。

In [27]:
@torch.no_grad()
def validate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    use_amp: bool = True,
    logger: logging.Logger | None = None,
) -> tuple[float, float]:
    model.eval()
    logger = logger or logging.getLogger("resnet_cifar10")
    amp_enabled = use_amp and device.type == "cuda"

    running_loss = 0.0
    running_correct = 0.0
    sample_count = 0

    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)
        with autocast(enabled=amp_enabled):
            logits = model(images)
            loss = criterion(logits, targets)

        batch_size = targets.size(0)
        running_loss += loss.item() * batch_size
        running_correct += (logits.argmax(dim=1) == targets).sum().item()
        sample_count += batch_size

    average_loss = running_loss / sample_count
    average_acc = running_correct / sample_count
    logger.info("val epoch done: loss=%.4f acc=%.4f", average_loss, average_acc)
    return average_loss, average_acc

## 7. Click 命令行入口

这一格把参数解析、日志、数据加载、模型初始化、checkpoint 恢复和完整训练循环串起来。需要从终端运行时，只要把 notebook 导出为 `.py` 或直接复用这里的 `main` 逻辑即可。

In [28]:
CSV_HISTORY_FIELDS = (
    "epoch",
    "train_loss",
    "train_acc",
    "val_loss",
    "val_acc",
    "lr",
    "best_val_acc",
)


def new_training_history() -> dict[str, list[float] | float]:
    return {
        "epoch": [],
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "lr": [],
        "best_val_acc": 0.0,
    }


def load_training_history_csv(history_csv_path: Path) -> dict[str, list[float] | float]:
    history = new_training_history()
    history_csv_path = Path(history_csv_path)
    if not history_csv_path.exists():
        return history

    with history_csv_path.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            history["epoch"].append(float(row["epoch"]))
            history["train_loss"].append(float(row["train_loss"]))
            history["train_acc"].append(float(row["train_acc"]))
            history["val_loss"].append(float(row["val_loss"]))
            history["val_acc"].append(float(row["val_acc"]))
            history["lr"].append(float(row["lr"]))
            history["best_val_acc"] = float(row["best_val_acc"])

    return history


def append_training_result_csv(
    history_csv_path: Path,
    epoch: int,
    train_loss: float,
    train_acc: float,
    val_loss: float,
    val_acc: float,
    lr: float,
    best_val_acc: float,
) -> None:
    history_csv_path = Path(history_csv_path)
    history_csv_path.parent.mkdir(parents=True, exist_ok=True)
    file_exists = history_csv_path.exists()

    with history_csv_path.open("a", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=CSV_HISTORY_FIELDS)
        if not file_exists:
            writer.writeheader()
        writer.writerow({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "lr": lr,
            "best_val_acc": best_val_acc,
        })


def plot_training_history(history_csv_path: Path, save_path: Path | None = None, show: bool = True) -> None:
    # 直接从 CSV 读取历史记录并画图，保证恢复训练时会沿用同一份历史。
    history = load_training_history_csv(history_csv_path)
    epochs = history.get("epoch", [])
    train_loss = history.get("train_loss", [])
    val_loss = history.get("val_loss", [])
    train_acc = history.get("train_acc", [])
    val_acc = history.get("val_acc", [])
    lr = history.get("lr", [])

    if not epochs:
        print("没有可视化的数据，请先完成一次训练。")
        return

    figure, axes = plt.subplots(1, 3, figsize=(16, 4))

    axes[0].plot(epochs, train_loss, label="train loss")
    axes[0].plot(epochs, val_loss, label="val loss")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("epoch")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, train_acc, label="train acc")
    axes[1].plot(epochs, val_acc, label="val acc")
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("epoch")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(epochs, lr, label="lr")
    axes[2].set_title("Learning Rate")
    axes[2].set_xlabel("epoch")
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    figure.tight_layout()
    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        figure.savefig(save_path, dpi=200, bbox_inches="tight")
    if show:
        plt.show()
    else:
        plt.close(figure)

In [29]:
@click.command()
@click.option("--input-dir", type=click.Path(path_type=Path, exists=True, file_okay=False, dir_okay=True), default=DEFAULT_INPUT_DIR, show_default=True, help="包含 batches.meta 和 batch 文件的原始 CIFAR-10 目录。")
@click.option("--export-dir", type=click.Path(path_type=Path, file_okay=False, dir_okay=True), default=DEFAULT_EXPORT_DIR, show_default=True, help="导出的图片目录。")
@click.option("--index-dir", type=click.Path(path_type=Path, file_okay=False, dir_okay=True), default=DEFAULT_INDEX_DIR, show_default=True, help="train/val 索引文件输出目录。")
@click.option("--prepare-data/--no-prepare-data", default=False, show_default=True, help="是否重新导出图片并生成索引文件；默认直接复用已有数据。")
@click.option("--epochs", type=int, default=DEFAULT_EPOCHS, show_default=True, help="训练轮数。")
@click.option("--batch-size", type=int, default=DEFAULT_BATCH_SIZE, show_default=True, help="每个 batch 的样本数。")
@click.option("--image-size", type=int, default=DEFAULT_IMAGE_SIZE, show_default=True, help="输入图像尺寸。")
@click.option("--lr", type=float, default=DEFAULT_LR, show_default=True, help="学习率。")
@click.option("--weight-decay", type=float, default=DEFAULT_WEIGHT_DECAY, show_default=True, help="权重衰减。")
@click.option("--num-workers", type=int, default=DEFAULT_NUM_WORKERS, show_default=True, help="DataLoader worker 数量。")
@click.option("--seed", type=int, default=DEFAULT_SEED, show_default=True, help="随机种子。")
@click.option("--use-amp/--no-use-amp", default=DEFAULT_USE_AMP, show_default=True, help="是否启用 AMP 混合精度训练。")
@click.option("--grad-clip-norm", type=float, default=DEFAULT_GRAD_CLIP_NORM, show_default=True, help="梯度裁剪阈值，0 表示不裁剪。")
@click.option("--log-dir", type=click.Path(path_type=Path, file_okay=False, dir_okay=True), default=DEFAULT_LOG_DIR, show_default=True, help="日志输出目录。")
@click.option("--history-csv-path", type=click.Path(path_type=Path, file_okay=True, dir_okay=False), default=DEFAULT_HISTORY_CSV_PATH, show_default=True, help="训练结果 CSV 文件路径。")
@click.option("--checkpoint-path", type=click.Path(path_type=Path, file_okay=True, dir_okay=False), default=DEFAULT_CHECKPOINT_PATH, show_default=True, help="训练 checkpoint 保存路径。")
@click.option("--resume-path", type=click.Path(path_type=Path, file_okay=True, dir_okay=False), default=None, help="恢复训练的 checkpoint 路径。")
def main(input_dir: Path, export_dir: Path, index_dir: Path, prepare_data: bool, epochs: int, batch_size: int, image_size: int, lr: float, weight_decay: float, num_workers: int, seed: int, use_amp: bool, grad_clip_norm: float, log_dir: Path, history_csv_path: Path, checkpoint_path: Path, resume_path: Path | None) -> dict[str, list[float] | float]:
    set_seed(seed)
    logger = setup_logging(log_dir=log_dir)
    device = get_device()
    amp_enabled = use_amp and device.type == "cuda"
    logger.info("使用设备: %s", device)
    logger.info("AMP enabled: %s", amp_enabled)
    logger.info("Gradient clip norm: %.4f", grad_clip_norm)

    history_csv_path = Path(history_csv_path)
    if resume_path is None and history_csv_path.exists():
        history_csv_path.unlink()
        logger.info("已清理旧的训练结果 CSV: %s", history_csv_path)

    export_counts: dict[str, int] | None = None
    index_counts: dict[str, int] | None = None
    if prepare_data:
        export_counts, index_counts = prepare_data_pipeline(
            input_dir=input_dir,
            export_dir=export_dir,
            index_dir=index_dir,
            pattern=DEFAULT_PATTERN,
            include_test=DEFAULT_INCLUDE_TEST,
            seed=seed,
            logger=logger,
        )
        logger.info("导出统计: %s", export_counts)
        logger.info("索引统计: %s", index_counts)
    else:
        train_index = index_dir / "train.txt"
        val_index = index_dir / "val.txt"
        if not export_dir.exists() or not train_index.exists() or not val_index.exists():
            raise FileNotFoundError(
                f"未找到已准备好的数据: export_dir={export_dir}, train_index={train_index}, val_index={val_index}. "
                "如果这是首次运行，请把 prepare_data 打开。"
            )
        logger.info("跳过数据导出和索引生成，直接复用已有数据。")

    train_loader, val_loader, class_names = build_dataloaders(
        image_root_dir=export_dir,
        index_dir=index_dir,
        batch_size=batch_size,
        image_size=image_size,
        num_workers=num_workers,
        logger=logger,
    )

    model = resnet18(num_classes=len(class_names)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(epochs, 1))
    scaler = GradScaler(enabled=amp_enabled)

    checkpoint_path = Path(checkpoint_path)
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    checkpoint_dir = checkpoint_path.parent
    best_val_acc = 0.0
    start_epoch = 1
    history = load_training_history_csv(history_csv_path) if resume_path is not None else new_training_history()

    if resume_path is not None:
        resume_path = Path(resume_path)
        if not resume_path.exists():
            raise FileNotFoundError(f"Checkpoint not found: {resume_path}")
        start_epoch, best_val_acc = load_checkpoint(
            checkpoint_path=resume_path,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler if amp_enabled else None,
            device=device,
            logger=logger,
        )
        if history_csv_path.exists():
            logger.info("已载入历史 CSV: %s", history_csv_path)
        else:
            logger.info("未找到历史 CSV，将从当前训练开始写入: %s", history_csv_path)

    for epoch in range(start_epoch, epochs + 1):
        logger.info("===== Epoch %s/%s =====", epoch, epochs)
        train_loss, train_acc = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device,
            scaler=scaler if amp_enabled else None,
            use_amp=amp_enabled,
            max_grad_norm=grad_clip_norm,
            logger=logger,
        )
        val_loss, val_acc = validate(
            model,
            val_loader,
            criterion,
            device,
            use_amp=amp_enabled,
            logger=logger,
        )
        scheduler.step()
        current_lr = scheduler.get_last_lr()[0]

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            save_checkpoint(
                checkpoint_path=checkpoint_path,
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                scaler=scaler if amp_enabled else None,
                epoch=epoch,
                best_val_acc=best_val_acc,
                logger=logger,
            )
            logger.info("本轮验证集指标刷新，已保存最优权重。")

        append_training_result_csv(
            history_csv_path=history_csv_path,
            epoch=epoch,
            train_loss=float(train_loss),
            train_acc=float(train_acc),
            val_loss=float(val_loss),
            val_acc=float(val_acc),
            lr=float(current_lr),
            best_val_acc=float(best_val_acc),
        )

        history["epoch"].append(float(epoch))
        history["train_loss"].append(float(train_loss))
        history["train_acc"].append(float(train_acc))
        history["val_loss"].append(float(val_loss))
        history["val_acc"].append(float(val_acc))
        history["lr"].append(float(current_lr))
        history["best_val_acc"] = float(best_val_acc)

        epoch_checkpoint_path = checkpoint_dir / f"checkpoint_{epoch}.pt"
        save_checkpoint(
            checkpoint_path=epoch_checkpoint_path,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler if amp_enabled else None,
            epoch=epoch,
            best_val_acc=best_val_acc,
            logger=logger,
        )

        plot_training_history(history_csv_path, save_path=Path(log_dir) / "training_history.png", show=False)

        logger.info(
            "epoch=%s train_loss=%.4f train_acc=%.4f val_loss=%.4f val_acc=%.4f lr=%.6f best_val_acc=%.4f",
            epoch,
            train_loss,
            train_acc,
            val_loss,
            val_acc,
            current_lr,
            best_val_acc,
        )

    logger.info("训练结束，最佳验证准确率: %.4f", best_val_acc)
    return history

In [30]:
# 训练入口：先跑 1 个 epoch，确认导出、索引、训练、验证和监控都能串起来。
history = main.callback(
    input_dir=DEFAULT_INPUT_DIR,
    export_dir=DEFAULT_EXPORT_DIR,
    index_dir=DEFAULT_INDEX_DIR,
    prepare_data=False,
    epochs=1,
    batch_size=32,
    image_size=DEFAULT_IMAGE_SIZE,
    lr=DEFAULT_LR,
    weight_decay=DEFAULT_WEIGHT_DECAY,
    num_workers=DEFAULT_NUM_WORKERS,
    seed=DEFAULT_SEED,
    use_amp=DEFAULT_USE_AMP,
    grad_clip_norm=DEFAULT_GRAD_CLIP_NORM,
    log_dir=DEFAULT_LOG_DIR,
    history_csv_path=DEFAULT_HISTORY_CSV_PATH,
    checkpoint_path=DEFAULT_CHECKPOINT_PATH,
    resume_path=None,
)

plot_training_history(DEFAULT_HISTORY_CSV_PATH, save_path=DEFAULT_HISTORY_PLOT_PATH)

2026-05-24 16:37:34,398 INFO 使用设备: cpu
2026-05-24 16:37:34,398 INFO AMP enabled: False
2026-05-24 16:37:34,398 INFO Gradient clip norm: 1.0000
2026-05-24 16:37:34,400 INFO 跳过数据导出和索引生成，直接复用已有数据。
2026-05-24 16:37:34,763 INFO 已加载索引数据集: image_root_dir=D:\Jupyter Lab\CiFar10\cifar-10-batches-py\export index_dir=D:\Jupyter Lab\CiFar10\dataset_index
2026-05-24 16:37:34,763 INFO 数据集划分完成: train=45000 val=5000
2026-05-24 16:37:34,763 INFO 类别数量: 10
C:\Users\34354\AppData\Local\Temp\ipykernel_17432\416165860.py:74: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=amp_enabled)
2026-05-24 16:37:34,811 INFO ===== Epoch 1/1 =====
C:\Users\34354\AppData\Local\Temp\ipykernel_17432\141519543.py:80: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp_enabled):


KeyboardInterrupt: 

## 8. 前向测试

这个单元只是一个轻量 smoke test，用随机输入检查 ResNet 的输出维度是否正常，方便快速确认前向路径没有写错。

In [ ]:
device = get_device()
set_seed(DEFAULT_SEED)
model = resnet18(num_classes=10).to(device)
dummy_input = torch.randn(2, 3, DEFAULT_IMAGE_SIZE, DEFAULT_IMAGE_SIZE, device=device)
dummy_output = model(dummy_input)
print("Smoke test output shape:", dummy_output.shape)

Smoke test output shape: torch.Size([2, 10])
